In [ ]:
%sql

SELECT
    run_date,
    total_execution_attempts,
    total_tables,
    successful_tables,
    failed_tables,
    skipped_tables,
    unknownStatus_tables,
    success_rate_pct,
    rowCountMismatch_tables,
    schemaChange_tables,
    total_bq_rows,
    total_loaded_rows,
    total_rowCountDifference,
    first_jobRun_ts,
    last_jobEnd_ts,
    total_elapsed_minutes

FROM prdrzranalytics.lab42.sdi_vw_pipelineMonitoring_gold_bqUcHealthSummary_daily

WHERE run_date = CURRENT_DATE();

In [ ]:
%sql

SELECT
    run_date,
    job_run_ts,
    job_end_ts,
    batch_number,
    bq_table,
    uc_table,
    run_type,
    status,
    bq_rows,
    loaded_rows,
    row_count_difference,
    elapsed_sec,
    error_message

FROM prdrzranalytics.lab42.sdi_vw_pipelineMonitoring_gold_bqUcFailedTables_daily

WHERE run_date = CURRENT_DATE()

ORDER BY
    batch_number,
    bq_table;

In [ ]:
%sql

WITH run_check AS (

    SELECT
        COUNT(*) AS run_count

    FROM prdrzranalytics.lab42.sdi_vw_pipelineMonitoring_gold_bqUcRunDetails_daily

    WHERE run_date = CURRENT_DATE()

)

SELECT
    assert_true(
        run_count > 0,

        CONCAT(
            'BQ to UC pipeline monitoring failed: ',
            'no pipeline executions were found for ',
            CURRENT_DATE(),
            '.'
        )
    ) AS pipeline_run_check

FROM run_check;

In [ ]:
%sql

WITH failure_summary AS (

    SELECT
        COUNT(*) AS failed_table_count

    FROM
        prdrzranalytics.lab42.sdi_vw_pipelineMonitoring_gold_bqUcFailedTables_daily

    WHERE run_date = CURRENT_DATE()

)

SELECT

    assert_true(

        failed_table_count = 0,

        CONCAT(
            'BQ to UC pipeline monitoring detected ',
            failed_table_count,
            ' failed table copy/copies for ',
            CURRENT_DATE(),
            '. Review ',
            'prdrzranalytics.lab42.',
            'sdi_vw_pipelineMonitoring_gold_bqUcFailedTables_daily ',
            'for failed table names and error details.'
        )

    ) AS failure_check

FROM failure_summary;